1. Using Calendarific API to get a full list of holidays, their name, date (in ISO format), description, and type.

In [30]:
import requests
import json
import pandas as pd

api_key = '49iY1MzO98nxEimi5h3Skg1yCqUI4XId'   # <--- Replace with your actual key!
url = 'https://calendarific.com/api/v2/holidays'

params = {
    'api_key': api_key,
    'country': 'US',       # Change to 'DE' for Germany, etc.
    'year': 2019,
    # 'type': 'national',  # Optional: only public/national holidays, remove for all
}

response = requests.get(url, params=params)
print('Status:', response.status_code)
data = response.json()

# Sample: See structure
print(json.dumps(data, indent=2))

# Extract holidays
holidays = data['response']['holidays']

# Save to JSON
with open('calendarific_holidays_2019_us.json', 'w') as f:
    json.dump(holidays, f, indent=2)

# Convert to DataFrame for analysis
df = pd.DataFrame(holidays)
print(df[['name', 'date', 'description']].head())

Status: 200
{
  "meta": {
    "code": 200
  },
  "response": {
    "holidays": [
      {
        "name": "New Year's Day",
        "description": "New Year's Day is the first day of the Gregorian calendar, which is widely used in many countries such as the USA.",
        "country": {
          "id": "us",
          "name": "United States"
        },
        "date": {
          "iso": "2019-01-01",
          "datetime": {
            "year": 2019,
            "month": 1,
            "day": 1
          }
        },
        "type": [
          "National holiday"
        ],
        "primary_type": "Federal Holiday",
        "canonical_url": "https://calendarific.com/holiday/us/new-year-day",
        "urlid": "us/new-year-day",
        "locations": "All",
        "states": "All"
      },
      {
        "name": "World Braille Day",
        "description": "World Braille Day celebrates the life and achievements of Louis Braille, who invented the braille code for the visually impaired.",
     

2. Normilizing data

In [31]:
with open('calendarific_holidays_2019_us.json') as f:
    holidays = json.load(f)

# Normalize
df_holidays = pd.json_normalize(
    holidays,
    sep='_',  # Separates nested keys with "_"
)

print(df_holidays.head())

                             name  \
0                  New Year's Day   
1               World Braille Day   
2                        Epiphany   
3          Orthodox Christmas Day   
4  International Programmers' Day   

                                         description  \
0  New Year's Day is the first day of the Gregori...   
1  World Braille Day celebrates the life and achi...   
2  Many people in the United States annually obse...   
3  Many Orthodox Christian churches in countries ...   
4  Many people celebrate International Programmer...   

                          type               primary_type  \
0           [National holiday]            Federal Holiday   
1  [United Nations observance]  United Nations observance   
2                  [Christian]                  Christian   
3                   [Orthodox]                   Orthodox   
4       [Worldwide observance]       Worldwide observance   

                                       canonical_url  \
0   https://calen

3. Flattening nested structure

In [32]:
with open('calendarific_holidays_2019_us.json') as f:
    holidays = json.load(f)

# Flatten the nested structure
df_holidays = pd.json_normalize(
    holidays,
    sep='_'
)

# Keep only relevant columns, and flatten the 'type' list
df_holidays = df_holidays[['name', 'description', 'date_iso', 'type']]
df_holidays['type'] = df_holidays['type'].apply(lambda x: ', '.join(x) if isinstance(x, list) else x)


In [37]:
h_include = [["national_holiday",'National holiday'],["christian_holiday",'Christian'],["muslim_holiday",'Muslim'],["worldwide_observance",'Worldwide observance']]
h_exclude = [['orthodox_holiday','Orthodox'],['hebrew_holiday','Hebrew'],['local_holiday','Local holiday']]

In [ ]:
df_temp = df_holidays

for i in h_include:
    
    df_temp[i[0]]=df_temp["type"].str.contains(i[1], case=False)

#for x in h_exclude:
 
    # df_temp.drop(df_temp["type"].str.contains(x[1]))

df_temp.reset_index(drop=True)



,name,description,date_iso,type,national_holiday,christian_holiday,muslim_holiday,worldwide_observance
0,New Year's Day,New Year's Day is the first day of the Gregori...,2019-01-01,National holiday,True,False,False,False
1,World Braille Day,World Braille Day celebrates the life and achi...,2019-01-04,United Nations observance,False,False,False,False
2,Epiphany,Many people in the United States annually obse...,2019-01-06,Christian,False,True,False,False
3,Orthodox Christmas Day,Many Orthodox Christian churches in countries ...,2019-01-07,Orthodox,False,False,False,False
4,International Programmers' Day,Many people celebrate International Programmer...,2019-01-07,Worldwide observance,False,False,False,True
...,...,...,...,...,...,...,...,...
576,Day After Christmas Day,Some states in the United States observe the D...,2019-12-26,Local holiday,False,False,False,False
577,Day After Christmas Day,Some states in the United States observe the D...,2019-12-26,Local holiday,False,False,False,False
578,Last Day of Chanukah,The last day of Hanukkah marks the end of a fe...,2019-12-30,Hebrew,False,False,False,False
579,New Year's Eve,New Year's Eve is the last day of the year in ...,2019-12-31,Observance,False,False,False,False


In [40]:
df_temp = df_filtered.groupby('date_iso', as_index=False).agg(lambda x: ', '.join(map(str, x)))


In [41]:
dates_2019 = pd.date_range(start="2019-01-01", end="2019-12-31", freq="D",)
df_dates_2019 = pd.DataFrame({"date_iso": dates_2019.astype(str)})
holiday_marks = df_temp.join(df_dates_2019.set_index('date_iso'),on='date_iso',how= 'right',)
holiday_marks.reset_index(inplace=True,drop=True)
holiday_marks["national_holiday"] = holiday_marks["national_holiday"].str.contains("True", case=False)
holiday_marks["christian_holiday"] = holiday_marks["christian_holiday"].str.contains("True", case=False)
holiday_marks["orthodox_holiday"] = holiday_marks["orthodox_holiday"].str.contains('True', case=False)
holiday_marks["hebrew_holiday"] = holiday_marks["hebrew_holiday"].str.contains('True', case=False)
holiday_marks["muslim_holiday"] = holiday_marks["muslim_holiday"].str.contains('True', case=False)
holiday_marks["worldwide_observance"] = holiday_marks["worldwide_observance"].str.contains("True", case=False)


4. Exporting to AWS database

In [ ]:
from sqlalchemy import create_engine, text

# 1. Set up connection (update credentials and host!)
from dotenv import dotenv_values

config = dotenv_values()

# define variables for the login
pg_user = config['POSTGRES_USER']  # align the key label with your .env file !
pg_host = config['POSTGRES_HOST']
pg_port = config['POSTGRES_PORT']
pg_db = config['POSTGRES_DB']
pg_schema = config['POSTGRES_SCHEMA']
pg_pass = config['POSTGRES_PASS']

url = f'postgresql://{pg_user}:{pg_pass}@{pg_host}:{pg_port}/{pg_db}'

engine = create_engine(url, echo=False)

# 2. Push your holidays DataFrame (assuming it's named df_holidays)
df_holidays.to_sql('holidays_2019_us', engine, schema=pg_schema, if_exists='replace', index=False)
holiday_marks.to_sql('holiday_marks', engine, schema=pg_schema, if_exists='replace', index=False)


365